### 01 - Instalações (Bibliotecas)

In [ ]:
%pip install scikit-learn

%pip install matplotlib

%pip install seaborn

%pip install pandas

!pip install torch torchvision

### 02 - Recursos (Importações)

In [ ]:
import re

from collections import Counter

from sklearn.model_selection import train_test_split

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sbn

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

### 03 - Classes

In [ ]:
class TextDataset:
  def __init__(self, X, y, vocabulary, max_lenght):
    self.X = [text_to_sequence(text, vocabulary, max_lenght) for text in X]

    self.y = y

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return torch.tensor(self.X[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.float32)

class AttentionLayer(nn.Module):
  def __init__(self, hidden_dimension):
    super(AttentionLayer, self).__init__()

    self.score_calculation = nn.Linear(hidden_dimension, 1)

  def forward(self, lstm_outputs):
    scores = self.score_calculation(lstm_outputs)

    attention_weights = F.softmax(scores, dim=1)

    weighted_vectors = lstm_outputs * attention_weights

    context_vector = torch.sum(weighted_vectors, dim=1)

    return context_vector, attention_weights

class HateSpeechAttentionModel(nn.Module):
  def __init__(self, vocabulary_size, embedding_dimension, hidden_dimension):
    super(HateSpeechAttentionModel, self).__init__()

    self.embedding = nn.Embedding(vocabulary_size, embedding_dimension, padding_idx=0)
    
    self.lstm = nn.LSTM(embedding_dimension, hidden_dimension, batch_first=True, bidirectional=True)

    self.attention = AttentionLayer(hidden_dimension * 2)

    self.dropout = nn.Dropout(0.3)

    self.fc = nn.Linear(hidden_dimension * 2, 1)

    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    embedded = self.dropout(self.embedding(x))

    lstm_output, _ = self.lstm(embedded)

    context_vector, attention_weights = self.attention(lstm_output)

    output = self.fc(self.dropout(context_vector))
    output = self.sigmoid(output)

    return output.squeeze(-1), attention_weights

### 04 - Funções

In [ ]:
def clean_text(text: str):
  normalized_text = re.sub(r'[^\w\s]', '', text.lower())

  return normalized_text

def text_to_sequence(text: str, vocabulary: Counter, max_lenght: int):
  tokens = clean_text(text).split()

  sequence = [vocabulary.get(token, vocabulary["<UNK>"]) for token in tokens]

  if (len(sequence) < max_lenght):
    sequence = sequence + ([vocabulary['<PAD>']] * (max_lenght - len(sequence)))
  else:
    sequence = sequence[:max_lenght]

  return sequence

def calculate_accuracy(predictions, labels):
  normalized_predictions = torch.round(predictions)

  correct_predictions = (normalized_predictions == labels).float()

  accuracy = correct_predictions.sum() / len(correct_predictions)

  return accuracy

# Usada para gerar o gráfico.

def plot_attention_map(tokens, attention_weights):
  weights = attention_weights.cpu().numpy().squeeze()

  plt.figure(figsize=(10,2))

  sbn.heatmap(
    [weights],
    xticklabels=tokens,
    yticklabels=["Atenção"],
    cmap="Reds",
    annot=True,
    fmt=".2f",
    cbar=False
  )

  plt.title("Mapa de Atenção (Frase)", pad=15)

  plt.xticks(rotation=45, ha="right", fontsize=11)
  plt.yticks(rotation=0, fontsize=11)

  plt.tight_layout()

  plt.show()

# Usada para detectar discurso de ódio.

def detect_hatred_with_attention(phrase: str, model: HateSpeechAttentionModel, vocabulary: Counter, inverse_vocabulary: Counter, max_lenght: int):
  model.eval()

  sequence = text_to_sequence(phrase, vocabulary, max_lenght)

  input_tensor = torch.tensor([sequence], dtype=torch.long).to(device)

  with torch.no_grad():
    prediction, attention_weights = model(input_tensor)

    prediction_value = prediction.item()

  result = ""

  if (prediction > 0.5):
    result = "Classificado como discurso de ódio!"
  else:
    result = "Classificado como um texto comum."

  print(f"\nFrase: {phrase}")

  print(f"\nPredição (Bruta): {prediction_value:.4f}")

  print(f"\nClassificação: {result}")

  real_tokens = [inverse_vocabulary.get(idx, "<UNK>") for idx in sequence]

  plot_attention_map(real_tokens, attention_weights[0])

### 05 - Implementação

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataframe = pd.read_csv("./Data/HateBR.csv", sep=",", usecols=["comentario", "label_final"])

print(dataframe.head())

X = dataframe["comentario"].values

Y = dataframe["label_final"].values

X_train, X_test, y_train, t_test = train_test_split(X, Y, test_size=0.25, random_state=42)

print(f"\nTamanho (Treino): {len(X_train)} amostras.")

print(f"\nTamanho (Teste): {len(X_test)} amostras.")

MAX_LENGHT = 15

all_words = [word for phrase in X_train for word in phrase.lower().split()]

words_frequency = Counter(all_words)

vocabulary = { word: idx + 2 for idx, (word, _) in enumerate(words_frequency.most_common()) }

vocabulary["<PAD>"] = 0
vocabulary["<UNK>"] = 1

inverse_vocabulary = { idx: word for word, idx in vocabulary.items() }

vocabulary_lenght = len(vocabulary)

print(f"\nTamanho (Vocabulário): {vocabulary_lenght}.")

train_dataset = TextDataset(X_train, y_train, vocabulary, max_lenght=MAX_LENGHT)

test_dataset = TextDataset(X_train, y_train, vocabulary, max_lenght=MAX_LENGHT)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, pin_memory=True)

model = HateSpeechAttentionModel(vocabulary_size=vocabulary_lenght, embedding_dimension=16, hidden_dimension=32).to(device)

print(f"\nModelo: {model}")

### 06 - Treinamento e Teste

In [ ]:
print("Treinamento:")

optimizer = optim.Adam(model.parameters(), lr=0.005)

criterion = nn.BCELoss()

epochs = 100

for epoch in range(0, epochs):
  model.train()

  train_loss = 0.0

  train_accurancy = 0.0

  for texts, labels in train_loader:
    texts, labels = texts.to(device), labels.to(device)

    optimizer.zero_grad()

    predictions, _ = model(texts)

    loss = criterion(predictions, labels)

    accuracy = calculate_accuracy(predictions, labels)

    loss.backward()

    optimizer.step()

    train_loss += loss.item()

    train_accurancy += accuracy.item()

  train_loss_average = train_loss / len(train_loader)

  train_accuracy_average = train_accurancy / len(train_loader)

  if ((epoch + 1) % 2 == 0) or (epoch == 0):
    print(f"\nÉpoca {epoch} de {epochs} | Perda: {train_loss_average:.4f} | Acurácia: {(train_accuracy_average * 100):.2f}%")

print("\nTeste:")

model.eval()

test_loss = 0.0

test_accuracy = 0.0

with torch.no_grad():
  for texts, labels in test_loader:
    texts, labels = texts.to(device), labels.to(device)

    predictions, _ = model(texts)

    loss = criterion(predictions, labels)

    accuracy = calculate_accuracy(predictions, labels)

    test_loss += loss.item()
    
    test_accuracy += accuracy.item()

  test_loss_average = test_loss / len(test_loader)
  
  test_accuracy_average = test_accuracy / len(test_loader)

  print(f"\nPerda: {test_loss_average:.4f} | Acurácia: {(test_accuracy_average * 100):.2f}%")

### 07 - Inferência Prática

In [ ]:
phrases = [
  "que dia excelente para estudar programacao",
  "eu odeio quando acaba a luz",
  "eu odeio pessoas assim, do seu partido, voces sao todos uns lixos",
  "meu dia fica pessimo quando esqueco o guarda-chuva",
  "odeio quando nao vou a fatec",
  "vou ficar muito triste quando me formar",
  "vou sentir saudades das aulas da fatec"
]

for phrase in phrases:
  detect_hatred_with_attention(phrase, model, vocabulary, inverse_vocabulary, max_lenght=MAX_LENGHT)